# Data Preparation
here you can find all the data preparation

### Tracking Data Alignment

The Automatic Tracking Data (ATD) and the final submission file use different frame counting conventions. The ATD `frame_id` is 1-indexed (starts at 1), while the tactical video frames required for the submission are 0-indexed (starts at 0)[cite: 15]. 

To align the tracking data with the submission format, a simple subtraction is required:
`video_frame = frame_id - 1`[cite: 3, 15]. 

This guarantees that row *i* of our prediction perfectly matches frame *i* of the tactical video, avoiding hard errors or silent offset penalties during evaluation[cite: 3, 5, 13].

In [1]:
import os
import sys
import pandas as pd
from kloppy import metrica
# Agregar la raíz al path para usar las utilidades proporcionadas
sys.path.insert(0, os.path.abspath('..'))
from games import asset_paths

In [4]:
def load_aligned_tracking(limit=None):
    """
    Carga los datos de seguimiento (ATD) y los alinea con los fotogramas del video.
    """
    # Obtener rutas de la cámara táctica (la referencia obligatoria)
    paths = asset_paths(source='tactical')
    
    # Cargar los datos crudos usando kloppy
    dataset = metrica.load_tracking_epts(
        meta_data=paths['atd_meta'], 
        raw_data=paths['atd_raw'], 
        limit=limit
    )
    
    # Convertir a DataFrame de Pandas
    df = dataset.to_df()
    
    # APLICAR LA ALINEACIÓN: video_frame = frame_id - 1
    df['video_frame'] = df['frame_id'] - 1
    
    # Mover video_frame al principio y configurarlo como índice
    # Esto facilitará mucho la inyección del Smart Tagging más adelante
    cols = ['video_frame'] + [c for c in df.columns if c != 'video_frame']
    df = df[cols]
    df.set_index('video_frame', inplace=True)
    
    return df

# Prueba de ejecución rápida (cargando solo los primeros 1000 frames)
if __name__ == "__main__":
    print("Cargando y alineando datos...")
    df_aligned = load_aligned_tracking(limit=1000)
    
    print("\nPrimeras filas del dataset alineado:")
    print(df_aligned[['frame_id', 'ball_x', 'ball_y']].head())
    print(f"\nNota cómo el índice 'video_frame' empieza en 0, mientras 'frame_id' empieza en 1.")

Cargando y alineando datos...

Primeras filas del dataset alineado:
             frame_id  ball_x  ball_y
video_frame                          
0                   1     NaN     NaN
1                   2     NaN     NaN
2                   3     NaN     NaN
3                   4     NaN     NaN
4                   5     NaN     NaN

Nota cómo el índice 'video_frame' empieza en 0, mientras 'frame_id' empieza en 1.


C:\Users\usuario\AppData\Local\Temp\ipykernel_27476\2682932651.py:25: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.set_index('video_frame', inplace=True)


## Handling Missing Ball Data: The Hybrid Approach

A critical challenge in the dataset is the absence of ball tracking coordinates on 41% of the graded frames. Instead of forcing raw physics-based imputations (which risk injecting misleading linear momentum into chaotic contested phases), we implement a **Hybrid Imputation Strategy**. This approach cleans minor tracking artifacts while feeding structural uncertainty directly into the neural network's Self-Attention mechanism.

Our strategy is broken down into four core components:

### 1. Micro-Interpolation for Short Gaps (Base Cleaning)
The ball rarely changes its trajectory abruptly within a fraction of a second without human interaction. Therefore, we safely interpolate very short gaps.
*   **Implementation:** We apply mathematical interpolation (e.g., Akima or Cubic spline) strictly for gaps lasting $< 25$ frames ($< 1$ second). 
*   **Rationale:** This fixes transient tracking failures and occlusions without fabricating "ghost passes" over long durations.

### 2. Blindness Flagging (The Missingness Channel)
For gaps lasting $\ge 25$ frames, we do **not** overwrite the `ball_x` and `ball_y` columns with heuristic guesses. Overwriting true missing data with flawed heuristics corrupts the training signal.
*   **Implementation:** We leave the coordinates as null (or a designated padding value) and introduce a new boolean feature: `ball_is_missing = 1`.
*   **Rationale:** By adding this flag, we explicitly teach the Temporal Convolutional Network (TCN) that it is "blind" at this timestep, forcing the network's attention weights to shift away from the ball and focus entirely on player interactions.

### 3. Heuristic Centroids as Auxiliary Features
When the ball disappears during prolonged play, it is typically occluded by a cluster of players contesting it. We leverage the findings from Borghesi et al. (2023), which show that the 5 nearest players (5NN) provide enough spatial context to infer possession.
*   **Implementation:** Instead of using the player centroid as a fake ball coordinate, we calculate the geometric center $(centroid\_x, centroid\_y)$ and the average velocity of the 3 to 5 players closest to the ball's last known location. We feed these as **new, independent features** to the network.
*   **Rationale:** The Self-Attention mechanism can learn to weigh this centroid data dynamically. If the centroid behaves like a ball in dispute, the network uses it; if the players scatter, the network learns to ignore it.

### 4. Strategic Abstention (The `X` Label Shield)
The official ranking metric, `spellF1`, evaluates full possession intervals rather than frame-by-frame accuracy. Crucially, any `X` (abstention) predicted during a graded frame is forward-filled with the last valid label.
*   **Implementation:** We tune the network's classification threshold so that prolonged periods of `ball_is_missing = 1` coupled with high spatial entropy (unclear player centroid) trigger a collapse to the `X` class.
*   **Rationale:** Emitting an `X` during heavy occlusions acts as a metric shield. It prevents the model from "flickering" between `H` and `A`, which would artificially split a continuous possession into multiple tiny, penalized spells.